In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('default') 
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [8]:

import ee
# ee.Authenticate(force=True)
ee.Initialize()

# -----------------------------
# Settings
# -----------------------------
YEAR = 2022
N_POINTS = 10000
EXPORT_DESC = f"us_annual_solar_avg_{YEAR}"
SCALE_METERS = 9000   # ERA5-Land native resolution is ~11 km, so 9-11 km is reasonable

# -----------------------------
# Contiguous US bounding box
# -----------------------------
conus_bbox = ee.Geometry.Rectangle([-125.0, 24.0, -66.5, 49.5], geodesic=False)

# -----------------------------
# Random sample points
# -----------------------------
points = ee.FeatureCollection.randomPoints(
    region=conus_bbox,
    points=N_POINTS,
    seed=42,
    maxError=1000
)

# Add lat/lon properties
def add_latlon(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        "lon": coords.get(0),
        "lat": coords.get(1),
        "year": YEAR
    })

points = points.map(add_latlon)

# -----------------------------
# Annual mean solar radiation
# -----------------------------
solar_ic = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01")
    .select("surface_solar_radiation_downwards")
)

annual_mean_solar = solar_ic.mean().rename("annual_mean_solar")

# -----------------------------
# Elevation
# -----------------------------
elevation = ee.Image("USGS/SRTMGL1_003").rename("elevation")

# Combine predictor/target bands
combined = annual_mean_solar.addBands(elevation)

# -----------------------------
# Sample image at points
# -----------------------------
samples = combined.sampleRegions(
    collection=points,
    properties=["lat", "lon", "year"],
    scale=SCALE_METERS,
    geometries=True
)

# -----------------------------
# Export to Drive as CSV
# -----------------------------
task = ee.batch.Export.table.toDrive(
    collection=samples,
    description=EXPORT_DESC,
    fileNamePrefix=EXPORT_DESC,
    fileFormat="CSV"
)

task.start()
print(f"Started export task: {EXPORT_DESC}")

Started export task: us_annual_solar_avg_2022


In [13]:

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# =========================================================
# 1. LOAD DATA
# =========================================================
# Change this to your actual file name/path
csv_path = "us_annual_solar_avg_2022.csv"

df = pd.read_csv(csv_path)

# =========================================================
# 2. BASIC CLEANING
# =========================================================
# Keep only the columns we need
# Change these names if your CSV uses slightly different column names
required_columns = ["lat", "lon", "elevation", "annual_mean_solar"]

missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}\nFound columns: {list(df.columns)}")

df = df[required_columns].copy()

# Drop missing rows
df = df.dropna()

# Keep only positive solar values
df = df[df["annual_mean_solar"] > 0]

if len(df) < 10:
    raise ValueError("Not enough valid rows after cleaning.")

# =========================================================
# 3. FEATURES / TARGET
# =========================================================
X = df[["lat", "lon", "elevation"]]
y = df["annual_mean_solar"]

# =========================================================
# 4. TRAIN / TEST SPLIT
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================================
# 5. TRAIN XGBOOST MODEL
# =========================================================
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

# =========================================================
# 6. EVALUATE
# =========================================================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model performance on test set:")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")

# =========================================================
# 7. EXAMPLE PREDICTION
# =========================================================
# Example point: lat, lon, elevation
sample_point = pd.DataFrame([{
    "lat": 39.74,
    "lon": -104.99,
    "elevation": 1600
}])

predicted_value = model.predict(sample_point)[0]
print(f"\nPredicted annual mean solar radiation for sample point: {predicted_value:.4f}")

# =========================================================
# 8. SAVE MODEL
# =========================================================
joblib.dump(model, "solar_xgboost_model.pkl")
print("\nSaved model to solar_xgboost_model.pkl")

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 47.8 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
Model performance on test set:
MAE:  117910.7464
RMSE: 484014.9059
R^2:  0.8840

Predicted annual mean solar radiation for sample point: 4887486.5000

Saved model to solar_xgboost_model.pkl
